In [1]:
# NAME = "Hariharan Mohan"
# WSU_ID = "B523W754"

In [2]:
# Importing Necessary libraries
import os
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Flatten, Dropout, UpSampling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import MeanIoU, Accuracy
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.preprocessing import LabelEncoder
# importing necessary libraries and modules for building, training, and visualizing deep neural networks using TensorFlow and Keras.

In [3]:
# Data paths
train_images_dir = r"C:\Users\ranja\Desktop\ML\train\Images"
train_masks_dir = r"C:\Users\ranja\Desktop\ML\train\Masks"
ground_truth_file = r"C:\Users\ranja\Desktop\ML\train\GroundTruth.csv"
seg_test_dir = r"C:\Users\ranja\Desktop\ML\test\ISBI2016_ISIC_Part1_Test_Data"
class_test_dir = r"C:\Users\ranja\Desktop\ML\test\ISIC2018_Task3_Test_Input"
# These paths are crucial for loading and organizing your dataset during the data loading and preprocessing steps.
#They allow your code to find and access the necessary files (images, masks, and ground truth data) needed for training and processing.

In [4]:
# Load ground truth data
ground_truth_data = pd.read_csv(ground_truth_file)
# This uses the read_csv function from the pandas library (pd) to read the contents of a CSV file specified by the variable ground_truth_csv_path

In [5]:
# Load and preprocess segmentation data
def load_segmentation_data(image_files, mask_files, target_height, target_width):
    images = []
    masks = []

    for file in image_files:
        img = cv2.imread(os.path.join(train_images_dir, file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (target_width, target_height))  # Resize the image
        img = img.astype('float32') / 255.0  # Normalize
        images.append(img)
        image = cv2.imread('your_image.jpg')
        

    for file in mask_files:
        mask = cv2.imread(os.path.join(train_masks_dir, file), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (target_width, target_height))  # Resize the mask
        mask = mask.astype('float32') / 255.0  # Normalize
        masks.append(mask)
        
        

    return np.array(images), np.array(masks)

In [6]:
target_height = 128
target_width = 128

In [7]:
# Load and preprocess segmentation data
def load_classification_data(image_files, mask_files, target_height, target_width):
    images = []
    masks = []

    for file in image_files:
        img = cv2.imread(os.path.join(train_images_dir, file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (target_width,target_height))  # Resize the image
        img = img.astype('float32') / 255.0  # Normalize
        
        images.append(np.expand_dims(img, axis=-1))  # 
      

        


    for file in mask_files:
       mask = cv2.imread(os.path.join(train_masks_dir, file), cv2.IMREAD_GRAYSCALE)
       mask = cv2.resize(mask, (128, 128))  # Resize the mask
       mask = mask.astype('float32') / 255.0  # Normalize
       masks.append(np.expand_dims(mask, axis=-1))  

    return np.array(images), np.array(masks)

In [8]:
def load_segmentation_test_data(test_files, target_height, target_width):
    test_images_seg = []
    for file in test_files:
        img = cv2.imread(os.path.join(seg_test_dir, file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (target_width, target_height))
        img = img.astype('float32') / 255.0
        test_images_seg.append(img)
    return np.array(test_images_seg)

# Load and preprocess classification test data
def load_classification_test_data(test_files, target_height, target_width):
    test_images_class = []
    for file in test_files:
        img = cv2.imread(os.path.join(class_test_dir, file), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (target_width, target_height))
        img = img.astype('float32') / 255.0
        test_images_class.append(img)
        #label = get_label_from_ground_truth(file)  # Retrieve labels
        #test_labels_class.append(label)
    return np.array(test_images_class)


In [9]:
# Assuming you've loaded images and masks using load_segmentation_data function

target_height = 128
target_width = 128


# Load images and masks
image_files = os.listdir(train_images_dir)
mask_files = os.listdir(train_masks_dir)

# Load and preprocess segmentation data
images, masks = load_segmentation_data(image_files, mask_files, target_height, target_width)

# Split data into training and validation sets
train_images_seg, val_images_seg, train_masks_seg, val_masks_seg = train_test_split(images, masks, test_size=0.2, random_state=42)


In [40]:
seg_test_files = os.listdir(seg_test_dir)
class_test_files = os.listdir(class_test_dir)

# Load segmentation test data
test_images_seg = load_segmentation_test_data(seg_test_files, target_height, target_width)

test_images_seg

array([[[0.88235295, 0.87058824, 0.8627451 , ..., 0.8627451 ,
         0.8627451 , 0.85490197],
        [0.8666667 , 0.87058824, 0.8666667 , ..., 0.8666667 ,
         0.8666667 , 0.8509804 ],
        [0.8666667 , 0.87058824, 0.8666667 , ..., 0.8666667 ,
         0.8666667 , 0.85490197],
        ...,
        [0.8980392 , 0.89411765, 0.8980392 , ..., 0.8784314 ,
         0.8784314 , 0.8901961 ],
        [0.8980392 , 0.89411765, 0.8980392 , ..., 0.8745098 ,
         0.8784314 , 0.8862745 ],
        [0.89411765, 0.9019608 , 0.9019608 , ..., 0.8352941 ,
         0.8509804 , 0.8627451 ]],

       [[0.08235294, 0.07058824, 0.10980392, ..., 0.22352941,
         0.19215687, 0.16862746],
        [0.07843138, 0.10196079, 0.12156863, ..., 0.21176471,
         0.1882353 , 0.15686275],
        [0.10980392, 0.11764706, 0.13725491, ..., 0.25490198,
         0.21568628, 0.1764706 ],
        ...,
        [0.16862746, 0.21176471, 0.24313726, ..., 0.34117648,
         0.30588236, 0.27450982],
        [0.1

In [42]:
# Load classification test data
test_images_class = load_classification_test_data(class_test_files, target_height, target_width)

array([[[0.24313726, 0.29803923, 0.30980393, ..., 0.29803923,
         0.34509805, 0.33333334],
        [0.2509804 , 0.30588236, 0.32156864, ..., 0.3529412 ,
         0.35686275, 0.32941177],
        [0.25882354, 0.30980393, 0.32941177, ..., 0.38431373,
         0.3647059 , 0.33333334],
        ...,
        [0.43529412, 0.4117647 , 0.4509804 , ..., 0.44705883,
         0.36862746, 0.34901962],
        [0.45490196, 0.4627451 , 0.49411765, ..., 0.39215687,
         0.3647059 , 0.32941177],
        [0.4509804 , 0.4627451 , 0.48235294, ..., 0.36078432,
         0.34901962, 0.3137255 ]],

       [[0.80784315, 0.80784315, 0.8156863 , ..., 0.60784316,
         0.60784316, 0.6156863 ],
        [0.8117647 , 0.8156863 , 0.827451  , ..., 0.60784316,
         0.6117647 , 0.61960787],
        [0.8156863 , 0.8235294 , 0.83137256, ..., 0.6117647 ,
         0.60784316, 0.60784316],
        ...,
        [0.7607843 , 0.76862746, 0.78431374, ..., 0.84313726,
         0.84313726, 0.84313726],
        [0.7

In [12]:
# Define input dimensions and number of classes for segmentation model

input_shape_seg = (target_height, target_width, 1)  # Adjust according to your image dimensions
num_classes_seg = 1  # Considering binary segmentation (1 class for masks)

# Define the segmentation model (CNN-based)
def define_segmentation_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)

    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    pool1 = MaxPooling2D((2, 2))(conv1)

    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
    pool2 = MaxPooling2D((2, 2))(conv2)

    conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)
    pool3 = MaxPooling2D((2, 2))(conv3)

    up1 = UpSampling2D((2, 2))(pool3)
    conv4 = Conv2D(128, (3, 3), activation='relu', padding='same')(up1)

    up2 = UpSampling2D((2, 2))(conv4)
    conv5 = Conv2D(64, (3, 3), activation='relu', padding='same')(up2)

    up3 = UpSampling2D((2, 2))(conv5)
    decoded = Conv2D(num_classes, (3, 3), activation='sigmoid', padding='same')(up3)

    seg_model = Model(inputs=inputs, outputs=decoded)
    seg_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return seg_model

# Create the segmentation model
seg_model = define_segmentation_model(input_shape_seg, num_classes_seg)

# Compile the segmentation model
seg_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the segmentation model with your prepared data (images and masks)
seg_model.fit(train_images_seg, train_masks_seg, epochs=5, batch_size=32, validation_data=(val_images_seg, val_masks_seg))

Epoch 1/5
13/13 [==============================] - 25s 2s/step - loss: 0.6108 - accuracy: 0.6887 - val_loss: 0.5756 - val_accuracy: 0.7067
Epoch 2/5
13/13 [==============================] - 24s 2s/step - loss: 0.5301 - accuracy: 0.7237 - val_loss: 0.5087 - val_accuracy: 0.7067
Epoch 3/5
13/13 [==============================] - 24s 2s/step - loss: 0.5098 - accuracy: 0.7237 - val_loss: 0.5073 - val_accuracy: 0.7067
Epoch 4/5
13/13 [==============================] - 25s 2s/step - loss: 0.5068 - accuracy: 0.7237 - val_loss: 0.5011 - val_accuracy: 0.7067
Epoch 5/5
13/13 [==============================] - 24s 2s/step - loss: 0.5007 - accuracy: 0.7237 - val_loss: 0.4986 - val_accuracy: 0.7067


In [13]:
seg_pred = seg_model.predict(test_images_seg)


12/12 [==============================] - 5s 398ms/step


In [14]:


import numpy as np

def jaccard_index(test_images_seg, seg_pred):
    intersection = np.sum(np.logical_and(test_images_seg, seg_pred))
    union = np.sum(np.logical_or(test_images_seg, seg_pred))
    
    return intersection / union if union != 0 else 0.0

def calculate_jaccard_index(test_images_seg_list, seg_pred_list, threshold=0.5):
    num_samples = len(test_images_seg_list)
    total_iou = 0.0

    for i in range(num_samples):
        test_images_seg = test_images_seg_list[i]
        seg_pred = (test_images_seg_list[i] > threshold).astype(np.uint8)  # Applying threshold to predicted masks
        total_iou += jaccard_index(test_images_seg.flatten(), seg_pred.flatten())  # Flatten arrays for calculations

    avg_iou = total_iou / num_samples if num_samples != 0 else 0.0
    return avg_iou

# Assuming you have test_masks_seg as ground truth masks and seg_predictions as predicted masks
threshold = 0.5  # Set threshold for Jaccard index evaluation
seg_jaccard_index = calculate_jaccard_index(test_images_seg, seg_pred, threshold)
print("Segmentation Jaccard Index:", seg_jaccard_index)



Segmentation Jaccard Index: 0.843003300182198


In [15]:
import os
from sklearn.model_selection import train_test_split
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

target_height = 128
target_width = 128


# Load images and masks
image_files = os.listdir(train_images_dir)
mask_files = os.listdir(train_masks_dir)

# Load and preprocess segmentation data
images, masks = load_classification_data(image_files, mask_files,target_height,target_width)

# Split data into training and validation sets
train_images_cls, val_images_cls, train_masks_cls, val_masks_cls = train_test_split(images, masks, test_size=0.2, random_state=42)




In [54]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def create_classification_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))  # Corrected line
    model.add(Dense(num_classes, activation='softmax'))
    return model

   
input_shape = (128, 128, 1)
num_classes = 10

# Load data
images, masks = load_classification_data(image_files, mask_files, target_height, target_width)

# Create the classification model
history=classification_model = create_classification_model(input_shape, num_classes)

# Reshape masks to match the output shape of the model
masks_reshaped = masks.reshape(-1, num_classes)

# Compile the model



# Create the classification model
classification_model = create_classification_model(input_shape, num_classes)

optimizer = Adam(lr=0.0001)
history=classification_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


classification_model.fit(images, masks_reshaped, epochs=20, batch_size=32, validation_split=0.2)


# Fit the model

Epoch 1/20
13/13 [==============================] - 5s 346ms/step - loss: 0.6257 - accuracy: 0.0000e+00 - val_loss: 1.2376 - val_accuracy: 0.0100
Epoch 2/20
13/13 [==============================] - 4s 344ms/step - loss: 0.6347 - accuracy: 0.0050 - val_loss: 1.0253 - val_accuracy: 0.0000e+00
Epoch 3/20
13/13 [==============================] - 5s 387ms/step - loss: 0.6193 - accuracy: 0.0025 - val_loss: 0.8913 - val_accuracy: 0.0000e+00
Epoch 4/20
13/13 [==============================] - 5s 403ms/step - loss: 0.6211 - accuracy: 0.0625 - val_loss: 0.9649 - val_accuracy: 0.0100
Epoch 5/20
13/13 [==============================] - 6s 422ms/step - loss: 0.6284 - accuracy: 0.0025 - val_loss: 1.0609 - val_accuracy: 0.0100
Epoch 6/20
13/13 [==============================] - 5s 425ms/step - loss: 0.6250 - accuracy: 0.3800 - val_loss: 0.9329 - val_accuracy: 0.1100
Epoch 7/20
13/13 [==============================] - 5s 414ms/step - loss: 0.6111 - accuracy: 0.0150 - val_loss: 1.2060 - val_accuracy: 0

In [55]:
class_pred=classification_model.predict(test_images_class)

16/16 [==============================] - 1s 76ms/step


In [59]:

import numpy as np
from sklearn.metrics import accuracy_score

# Assuming your model is already trained and you have test data: test_images

# Make predictions on the test data
class_pred = classification_model.predict(test_images_class)

# Convert predictions to class indices
predicted_classes = np.argmax(class_pred, axis=1)

# Calculate accuracy for each class
class_accuracies = []
for class_idx in range(num_classes):
    correct = np.sum(predicted_classes == class_idx)
    total = len(predicted_classes)
    class_accuracy = correct / total
    class_accuracies.append(class_accuracy)

# Normalize accuracies
normalized_accuracy = np.mean(class_accuracies)


normalized_accuracy_percentage = normalized_accuracy * 100

# Print normalized accuracy as a percentage
print(f"Normalized Accuracy: {normalized_accuracy_percentage:.5f}%")

16/16 [==============================] - 1s 78ms/step
Normalized Accuracy: 10.00000%
